***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [6. 成像中的去卷积](6_0_introduction.ipynb)
    * 上一节： [6.5 源搜索与检测](6_5_source_finding.ipynb)
    * 下一节： [6.x 延伸阅读与参考文献](6_x_further_reading_and_references.ipynb)

***


## 6.6 正则化去卷积与残差统计

CLEAN 把天空表示为一组点源或有限尺度分量，并通过迭代减去 PSF 来逼近与数据一致的模型。这个思想足以支撑大量射电成像工作，但它也揭示了去卷积问题的本质：脏图像不是天空的直接照片，而是不完整 Fourier 采样、噪声、权重和 PSF 共同作用后的结果。去卷积必须在“符合数据”和“选择怎样的天空模型”之间取得平衡。

第 2.15 节已经把正则化和过拟合作为一般数学问题讨论。本节把这些语言带回去卷积。无论算法名叫 Högbom CLEAN、多尺度 CLEAN、最大熵、压缩感知、Bayesian imaging 还是联合校准成像，其核心都可以写成类似的反问题：

$$
\hat{x}=rg\min_x\left[(Ax-d)^\dagger C_n^{-1}(Ax-d)+\lambda R(x)ight],
$$

其中 $d$ 是可见度或脏图像数据，$A$ 是测量算子，$C_n$ 是噪声协方差，$R(x)$ 描述对天空模型的约束或先验。不同算法的差异，主要在于 $A$ 如何实现、$R(x)$ 选择了怎样的天空形态、以及 $\lambda$ 或停止准则怎样确定。

![正则化去卷积目标函数](figures/regularization_family_objective.png)

**图 6.6.1** 正则化去卷积可以看成数据一致性、模型先验和显式约束的组合。点源稀疏、多尺度、平滑最大熵、全变差和支撑区域约束代表不同的天空模型假设。

### 6.6.1 CLEAN 也是一种先验

点源 CLEAN 的先验很明确：天空可以由许多点分量近似，且每次只加入少量通量。循环增益限制了每一步模型增长，mask 限制了模型支撑区域，停止阈值限制了模型继续吸收残差的程度。多尺度 CLEAN 把点源字典扩展为有限尺度分量，因此更适合展源；MT-MFS 把宽带天空写成频率 Taylor 项，因此可以在连续谱成像中同时拟合强度和谱结构。

从正则化角度看，CLEAN 家族并不只是工程算法。它们把“天空应该由哪些基函数组成”“哪些区域可以有发射”“允许模型解释到多低的残差”这些先验写进了迭代流程。若真实天空与先验匹配，去卷积可以稳定恢复结构；若真实天空包含很弥散的低面亮度成分、复杂谱结构或方向相关残差，点源或简单尺度字典就会把不匹配的结构拆成伪分量。

### 6.6.2 残差不只是 RMS 数字

去卷积完成后，残图是判断模型是否可信的核心证据。理想残差应接近噪声经过成像权重和 PSF 相关化后的结果，而不是完全独立的白噪声。仅报告一个全图 RMS 容易遗漏空间结构：强源周围的相干旁瓣、扩展源边缘的正负环、primary beam 边缘的噪声放大、未建模方向相关误差和过度 CLEAN 后的噪声坑，都可能在相同 RMS 下具有完全不同的科学含义。

因此，残差统计至少应同时看三个层次。第一是幅度分布，例如归一化残差直方图是否接近对称、是否有长尾。第二是空间相关，例如残差自相关或局部 RMS 图是否显示大尺度结构。第三是与模型和 PSF 的关系，例如强模型分量附近是否还有系统性残差，残差峰是否沿 PSF 旁瓣方向排列。

![残差统计诊断](figures/residual_statistics_diagnostics.png)

**图 6.6.2** 残差诊断不能只看一个 RMS。左图是含有局部结构的残差图，中图比较归一化残差直方图与单位高斯分布，右图显示残差自相关截面。相干残差说明仍有未建模结构、校准误差或过拟合/欠拟合问题。

### 6.6.3 正则化强度与过拟合

正则化强度 $\lambda$ 或 CLEAN 停止阈值控制模型自由度。$\lambda$ 过大时，模型被先验压得过于简单，真实结构留在残差中；$\lambda$ 过小时，模型过度追随噪声和校准残差。CLEAN 中类似的现象表现为欠 CLEAN 和过 CLEAN：前者留下明显旁瓣和扩展结构残差，后者把噪声峰吸入模型，导致通量偏差、局部负坑和源表可靠性下降。

L-curve、残差统计、注入源恢复和保留可见度验证都可以帮助选择折中。保留可见度验证的思想是：不把所有数据都用于建模，而是留出一部分时间、频率或基线样本检查模型预测能力。若模型在训练数据上的残差持续下降，但在保留数据上的误差开始上升，就说明模型正在拟合噪声或数据特定伪影。

![L 曲线与验证误差](figures/lcurve_and_validation.png)

**图 6.6.3** 左图示意正则化问题中的 L-curve；右图示意保留数据验证误差随正则化强度变化。过弱正则化会提高模型自由度并增加过拟合风险，过强正则化则会欠拟合真实结构。

### 6.6.4 源表、通量和残差统计必须闭环

第 6.5 节讨论源搜索时已经看到，源表质量依赖局部 RMS、背景估计、波束模型和残差图。这里可以进一步说明：源搜索不是去卷积之后完全独立的后处理，而是对去卷积模型的一次外部审计。若过度 CLEAN 把噪声写入模型，源搜索会产生更多低信噪比伪源；若欠 CLEAN 留下强旁瓣，局部 RMS 会被抬高，真实弱源会被漏检；若扩展结构被错误拆成点源串，积分通量和形态参数都会偏离真实物理量。

因此，可靠的连续谱产品至少应同时交付 restored image、residual image、PSF、local RMS map、source catalogue 和参数/provenance 记录。只给一张恢复图像，不足以判断源表是否可信；只给源表而不看残差图，也无法区分真实源和成像伪影。

![过度去卷积与通量恢复](figures/overcleaning_flux_recovery_case.png)

**图 6.6.4** 去卷积停止准则会改变通量恢复和残差结构。停止过早会留下真实发射；折中解保留主要结构并避免吸收噪声；过度去卷积会把噪声和条纹写入模型，使源表和低面亮度测量变得不可靠。

### 6.6.5 教学案例：低面亮度晕的可信检测

考虑一个星系团低面亮度射电晕。自然权重和 taper 图像中似乎存在弥散发射，但均匀权重图像只显示若干嵌入点源。判断这个晕是否可信，不能只依赖一张深 CLEAN 图像。首先需要检查短基线是否覆盖目标角尺度，随后比较点源减除前后的 residual，确认弥散结构不只是点源旁瓣残留。接着改变多尺度列表、mask 和阈值，检查积分通量是否稳定。最后可注入类似尺度和通量的模拟晕，观察处理流程是否能恢复它。

若弥散通量只在某一组 mask 和阈值下出现，且残图中仍有与 PSF 对齐的结构，结论应降级为候选检测；若不同 taper、多尺度设置和注入源测试都支持类似通量，残差统计也没有显示系统性结构，检测可信度才逐步提高。这个案例把第 4 章的短基线约束、第 5 章的权重和参数选择、第 6 章的正则化去卷积以及第 9 章的处理报告自然连接起来。

### 6.6.6 本节结论

去卷积是带有先验和正则化的反问题。CLEAN 家族通过点源、多尺度、mask、循环增益和停止阈值实现一种可操作的正则化；其他方法则可能显式使用平滑性、稀疏性、熵、全变差或 Bayesian 先验。无论算法形式如何，模型可信度都必须由残差统计、通量稳定性、保留数据验证、注入源恢复和源表审计共同支持。残差变小不必然意味着科学结论更可靠；只有当残差结构、模型自由度和科学测量相互一致时，去卷积结果才真正可解释。

***

下一节： [6.x 延伸阅读与参考文献](6_x_further_reading_and_references.ipynb)
